In [2]:
import sys
from pathlib import Path
import pandas as pd

# notebooks 폴더에서 프로젝트 최상위 폴더를 찾기
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.installment import (
    get_installment_transactions,
    add_monthly_payment,
    calculate_remaining_installment,
    calculate_next_month_payment,
    summarize_next_month_by_customer
)

In [4]:
df = pd.read_csv(PROJECT_ROOT / "data" / "transactions_with_installment_clean.csv")

print(df.shape)
print(df.columns.tolist())

(318145, 16)
['transaction_id', 'customer_id', 'persona_id', 'age_group', 'gender', 'occupation', 'annual_income', 'household_type', 'transaction_datetime', 'merchant_name', 'category', 'amount', 'payment_method', 'card_product', 'transaction_status', 'installment_months']


In [5]:
installment_df = get_installment_transactions(df)
installment_df = add_monthly_payment(installment_df)
installment_df = calculate_remaining_installment(
    installment_df,
    "2025-09"
)
installment_df = calculate_next_month_payment(installment_df)

print("할부 거래 수:", len(installment_df))

installment_df[
    [
        "transaction_id",
        "customer_id",
        "amount",
        "installment_months",
        "monthly_payment",
        "paid_months",
        "remaining_months",
        "remaining_amount",
        "next_month_payment"
    ]
].head(10)

할부 거래 수: 1812


,transaction_id,customer_id,amount,installment_months,monthly_payment,paid_months,remaining_months,remaining_amount,next_month_payment
414,TX000415,C0784,484300,3,161433.333333,3,0,0.0,0.000000
482,TX000483,C0943,1577600,2,788800.000000,2,0,0.0,0.000000
523,TX000524,C0460,1488300,2,744150.000000,2,0,0.0,0.000000
665,TX000666,C0988,2126100,3,708700.000000,3,0,0.0,0.000000
677,TX000678,C0342,564700,2,282350.000000,2,0,0.0,0.000000
680,TX000681,C0942,664700,2,332350.000000,2,0,0.0,0.000000
1006,TX001007,C0646,451800,12,37650.000000,9,3,112950.0,37650.000000
1037,TX001038,C0096,530700,6,88450.000000,6,0,0.0,0.000000
1044,TX001045,C0523,453800,2,226900.000000,2,0,0.0,0.000000
1285,TX001286,C0089,442900,12,36908.333333,9,3,110725.0,36908.333333


In [7]:
customer_summary = summarize_next_month_by_customer(installment_df)

print("할부 고객 수:", len(customer_summary))

customer_summary.head(10)

할부 고객 수: 611


,customer_id,next_month_installment,remaining_installment_amount,installment_count
0,C0006,0.000000,0.000000e+00,1
1,C0007,0.000000,0.000000e+00,1
2,C0009,0.000000,0.000000e+00,1
3,C0010,0.000000,0.000000e+00,1
4,C0011,534783.333333,2.051567e+06,6
5,C0012,493950.000000,2.585917e+06,5
6,C0014,335175.000000,3.016575e+06,4
7,C0016,0.000000,0.000000e+00,1
8,C0017,0.000000,0.000000e+00,1
9,C0019,0.000000,0.000000e+00,3


In [8]:
top_customers = customer_summary.sort_values(
    "next_month_installment",
    ascending=False
).head(10)

top_customers

,customer_id,next_month_installment,remaining_installment_amount,installment_count
99,C0160,4.665625e+06,1.593281e+07,27
279,C0430,3.631167e+06,1.451216e+07,28
425,C0667,2.816900e+06,5.444400e+06,4
512,C0827,2.171083e+06,5.511000e+06,17
328,C0508,1.879442e+06,9.733225e+06,18
264,C0406,1.854075e+06,5.400925e+06,14
240,C0368,1.737958e+06,4.361075e+06,18
118,C0184,1.541400e+06,1.541400e+06,6
10,C0020,1.431000e+06,2.040467e+06,7
445,C0703,1.420567e+06,1.420567e+06,12


In [11]:
representative_customer = top_customers.iloc[0]["customer_id"]

representative_customer


'C0160'

In [12]:
customer_detail = installment_df[
    installment_df["customer_id"] == representative_customer
][
    [
        "transaction_id",
        "transaction_datetime",
        "merchant_name",
        "amount",
        "installment_months",
        "monthly_payment",
        "paid_months",
        "remaining_months",
        "remaining_amount",
        "next_month_payment"
    ]
].sort_values("transaction_datetime")

customer_detail

,transaction_id,transaction_datetime,merchant_name,amount,installment_months,monthly_payment,paid_months,remaining_months,remaining_amount,next_month_payment
9799,TX009800,2025-01-12 14:32:00,이마트,1061400,2,5.307000e+05,2,0,0.000000e+00,0.000000
11618,TX011619,2025-01-14 16:08:00,신세계백화점,560900,2,2.804500e+05,2,0,0.000000e+00,0.000000
12507,TX012508,2025-01-15 18:35:00,인터파크트리플,1564100,2,7.820500e+05,2,0,0.000000e+00,0.000000
29499,TX029500,2025-02-05 15:30:00,신세계백화점,1507300,2,7.536500e+05,2,0,0.000000e+00,0.000000
51178,TX051179,2025-03-02 17:58:00,이마트,478000,6,7.966667e+04,6,0,0.000000e+00,0.000000
51734,TX051735,2025-03-03 12:50:00,롯데백화점,559200,3,1.864000e+05,3,0,0.000000e+00,0.000000
94940,TX094941,2025-04-23 11:30:00,내과의원,377400,3,1.258000e+05,3,0,0.000000e+00,0.000000
100838,TX100839,2025-04-30 19:06:00,신세계백화점,1787000,12,1.489167e+05,6,6,8.935000e+05,148916.666667
126127,TX126128,2025-05-30 20:05:00,이마트,2414600,6,4.024333e+05,5,1,4.024333e+05,402433.333333
127636,TX127637,2025-06-01 20:47:00,갤러리아백화점,4439000,3,1.479667e+06,3,0,0.000000e+00,0.000000


In [14]:
customer_summary_row = customer_summary[
    customer_summary["customer_id"] == representative_customer
]

customer_summary_row

,customer_id,next_month_installment,remaining_installment_amount,installment_count
99,C0160,4665625.0,1.593281e+07,27


In [15]:
row = customer_summary_row.iloc[0]

print("고객:", row["customer_id"])
print("할부 건수:", int(row["installment_count"]))
print("다음 달 할부 예정액:", f'{row["next_month_installment"]:,.0f}원')
print("남은 할부금:", f'{row["remaining_installment_amount"]:,.0f}원')

고객: C0160
할부 건수: 27
다음 달 할부 예정액: 4,665,625원
남은 할부금: 15,932,808원
